IMPORT PACKAGES

In [31]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from ISLP import load_data
from sklearn.model_selection import train_test_split, cross_validate, cross_val_score
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression as LR
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier as RFC

LOAD DATA

In [32]:
nhis18 = pd.read_sas('nhis2018samadult.sas7bdat')
nhis18

,FPX,FMX,HHX,INTV_QRT,WTIA_SA,WTFA_SA,SEX,HISPAN_I,R_MARITL,MRACRPI2,...,DEP_2,PAIN_4,TIRED_1,TIRED_2,TIRED_3,RCS_AFD,PAIN_2A,ANX_3R,DEP_3R,COGCAUS2
0,1.0,1.0,1.0,1.0,4228.8,3915.0,2.0,12.0,4.0,1.0,...,2.0,NaN,1.0,NaN,NaN,4.0,1.0,1.0,NaN,NaN
1,1.0,1.0,6.0,1.0,14650.9,16978.0,1.0,12.0,1.0,1.0,...,2.0,NaN,2.0,1.0,2.0,4.0,1.0,2.0,1.0,NaN
2,1.0,1.0,8.0,1.0,7066.1,10385.0,1.0,12.0,1.0,1.0,...,2.0,3.0,2.0,2.0,3.0,4.0,4.0,NaN,NaN,NaN
3,1.0,1.0,9.0,1.0,4497.5,3958.0,1.0,12.0,4.0,11.0,...,2.0,2.0,2.0,1.0,3.0,4.0,4.0,2.0,3.0,7.0
4,1.0,1.0,10.0,1.0,5594.0,6483.0,1.0,12.0,1.0,1.0,...,2.0,1.0,1.0,NaN,NaN,4.0,4.0,3.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25412,1.0,3.0,55556.0,4.0,4469.8,7066.0,2.0,12.0,7.0,1.0,...,2.0,NaN,2.0,1.0,1.0,4.0,1.0,1.0,NaN,NaN
25413,1.0,1.0,55557.0,4.0,8575.3,11808.0,2.0,12.0,7.0,2.0,...,8.0,NaN,8.0,NaN,NaN,1.0,8.0,NaN,NaN,NaN
25414,1.0,1.0,55560.0,4.0,10446.3,9657.0,2.0,12.0,7.0,1.0,...,2.0,3.0,4.0,2.0,2.0,4.0,4.0,1.0,1.0,9.0
25415,1.0,1.0,55562.0,4.0,15979.8,16525.0,2.0,12.0,1.0,1.0,...,2.0,1.0,2.0,1.0,1.0,4.0,2.0,NaN,NaN,NaN


CLEAN DATA

In [33]:
#select features
cols = ['AASMEV', 'AASSTILL', 'DEP_2', 'ASISAD', 'SMKNOW', 'SMKEV', 'AUSUALPL', 
        'AHCDLYR1', 'AHCDLYR2', 'AHCDLYR3', 'AHCDLYR4', 'AHCDLYR5', 'SEX', 'AGE_P']
nhis18 = nhis18[cols]

#remove 85+ year olds
nhis18 = nhis18[nhis18['AGE_P'] < 85]

#create depression variable
nhis18['depression'] = np.where(
    (nhis18['DEP_2'] == 1) | (nhis18['ASISAD'].isin([1, 2, 3])), 1, #depressed
    np.where(
        (nhis18['DEP_2'] == 2) & (nhis18['ASISAD'].isin([4, 5])), 0, #not depressed
        np.nan))

#create asthma variable
nhis18['asthma'] = np.where(
    (nhis18['AASMEV'] == 1) & (nhis18['AASSTILL'] == 1), 1, #asthma
    np.where(
        nhis18['AASMEV'] == 2, 0, #no asthma
        np.nan))

#create smoking variable
nhis18['smoking'] = np.select(
    [(nhis18['SMKEV'] == 1) & (nhis18['SMKNOW'] == 3),
     (nhis18['SMKEV'] == 1) & (nhis18['SMKNOW'].isin([1, 2])),
     (nhis18['SMKEV'] == 2)],
    [2,  #former
     3,  #current
     1], #never
    default = np.nan)

#create healthcare accessibility variable
delay_cols = ['AHCDLYR1', 'AHCDLYR2', 'AHCDLYR3', 'AHCDLYR4', 'AHCDLYR5']

nhis18['any_delay'] = np.where(
    nhis18[delay_cols].isin([1]).any(axis=1), 1,
    np.where(
        nhis18[delay_cols].isin([2]).all(axis=1), 0,
        np.nan))

nhis18['health_access'] = np.where(
    (nhis18['AUSUALPL'].isin([1, 3])) & (nhis18['any_delay'] == 0), 0,  #accessible
    np.where(
        (nhis18['AUSUALPL'] == 2) | (nhis18['any_delay'] == 1), 1,  #inaccessible
        np.nan))

#recode sex variable
nhis18['female'] = np.where(nhis18['SEX'] == 2, 1, 0) 

#final dataset
nhis18_final = nhis18[['depression', 'asthma', 'smoking', 'health_access', 'AGE_P', 'female']].dropna()
nhis18_final

,depression,asthma,smoking,health_access,AGE_P,female
0,0.0,0.0,2.0,0.0,79.0,1
1,0.0,1.0,1.0,0.0,37.0,0
2,0.0,0.0,1.0,0.0,29.0,0
3,1.0,0.0,2.0,1.0,75.0,0
5,0.0,0.0,1.0,0.0,54.0,1
...,...,...,...,...,...,...
25410,0.0,0.0,1.0,0.0,20.0,1
25411,0.0,1.0,1.0,0.0,20.0,1
25412,0.0,0.0,1.0,0.0,19.0,1
25415,0.0,0.0,1.0,0.0,61.0,1


DEFINE X AND Y AND FEATURE TYPES

In [34]:
X = nhis18_final.drop(columns=['depression'])
y = nhis18_final['depression']

numeric = ['AGE_P']
binary = ['asthma', 'health_access', 'female']
categorical = ['smoking']

TEST/TRAIN SPLIT

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

PREPROCESSING

In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric),
        ('cat', OneHotEncoder(drop='first'), categorical),  
        ('bin', 'passthrough', binary)
    ]
)

MODEL PIPELINES

In [37]:
log_model = Pipeline([
    ("preprocess", preprocessor),
    ("log", LR(max_iter=1000))
     ])

svm_model = Pipeline([
    ("preprocess", preprocessor),
    ("svm", SVC(kernel="rbf", probability=True))
     ])

rf_model = Pipeline([
    ("preprocess", preprocessor),
    ("rf", RFC(n_estimators=200, random_state=0))
     ])

CROSS-VALIDATION

In [38]:
models = {
    "Logistic Regression": log_model,
    "SVM": svm_model,
    "Random Forest": rf_model
}

cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name}: Mean Accuracy = {scores.mean():.3f}, Std = {scores.std():.3f}")

Logistic Regression: Mean Accuracy = 0.807, Std = 0.001
SVM: Mean Accuracy = 0.807, Std = 0.001
Random Forest: Mean Accuracy = 0.793, Std = 0.003


Across the three models (Logistic Regression, SVM, and Random Forest), overall accuracy is fairly similar, ranging from about 0.80 to 0.81, with Logistic Regression (80.7%) and SVM (80.7%) performing slightly better than Random Forest (79.3%).

FEATURE IMPORTANCE

In [39]:
log_model.fit(X, y)

#get feature names
feature_names = (
    numeric +
    list(log_model.named_steps['preprocess']
         .named_transformers_['cat']
         .get_feature_names_out(categorical)) +
    binary
)

coefficients = log_model.named_steps['log'].coef_[0]

results = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Odds Ratio': np.exp(coefficients)
}).sort_values(by='Odds Ratio', ascending=False)

print(results)

         Feature  Coefficient  Odds Ratio
2    smoking_3.0     0.782913    2.187837
3         asthma     0.670136    1.954503
5         female     0.617302    1.853920
4  health_access     0.521575    1.684678
1    smoking_2.0     0.364996    1.440509
0          AGE_P     0.132231    1.141371


All selected features (smoking status, asthma status, sex, age, and healthcare accessibility) demonstrate a positive association with depression. Current smokers had more than twice the odds (OR = 2.18) of having depression compared to those who never smoked. Similarly, adults with asthma had nearly twice the odds (OR = 1.95) of depression compared to adults without asthma. 